## Importation des bibliothèques nécessaires :

In [55]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from huggingface_hub import notebook_login
import evaluate

notebook_login()

# Analyse de sentiments

## Charger les données

In [56]:
Sentiment_train = pd.read_parquet("hf://datasets/lelapa/SentimentTrain/data/train-00000-of-00001.parquet")
Sentiment_test = pd.read_parquet("hf://datasets/lelapa/SentimentTest/data/train-00000-of-00001.parquet")

## Normaliser les cibles (targets) pour les données d'entraînement

In [57]:
label_map = {'Korau': 0, 'Kyakkyawa': 1, 'Tsaka-tsaki': 2,
             'Hasi': 0, 'Chanya': 1, 'Wastani': 2}
Sentiment_train['targets'] = Sentiment_train['targets'].map(label_map)

In [58]:
# Supprimer la colonne `targets` de Sentiment_test
if 'targets' in Sentiment_test.columns:
    print(f"Suppression de la colonne 'targets' contenant {Sentiment_test['targets'].isna().sum()} valeurs manquantes.")
    Sentiment_test = Sentiment_test.drop(columns=['targets'])

Suppression de la colonne 'targets' contenant 300 valeurs manquantes.


## Charger le tokenizer et le modèle

In [59]:
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B", trust_remote_code=True)

## Définir le token de padding

In [60]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("lelapa/InkubaLM-0.4B", trust_remote_code=True)
model.eval()

VulavulaLlamaForCausalLM(
  (model): VulavulaLlamaModel(
    (embed_tokens): Embedding(61788, 2048)
    (layers): ModuleList(
      (0-7): 8 x VulavulaLlamaDecoderLayer(
        (self_attn): VulavulaLlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): VulavulaLlamaRotaryEmbedding()
        )
        (mlp): VulavulaLlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): VulavulaLlamaRMSNorm()
        (post_attention_layernorm): VulavulaLlamaRMSNorm()
  

## Fonction pour extraire les embeddings

In [61]:
def get_embeddings(texts, model, tokenizer, device):
    embeddings = []
    for text in tqdm(texts, desc="Extraction des embeddings"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Utiliser la dernière couche cachée moyenne comme embedding
            hidden_states = outputs.hidden_states[-1]
            sentence_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(sentence_embedding)
    return embeddings

In [62]:
# Préparer les données pour le modèle de classification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Extraire les embeddings pour les données d'entraînement
X_train = get_embeddings(Sentiment_train['inputs'], model, tokenizer, device)
y_train = Sentiment_train['targets']

# Extraire les embeddings pour les données de test
X_test = get_embeddings(Sentiment_test['inputs'], model, tokenizer, device)

# Entraîner un modèle de classification (Logistic Regression)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Faire des prédictions sur les données de test
y_pred = clf.predict(X_test)

# Sauvegarder les prédictions dans le dataset de test
Sentiment_test['predictions'] = y_pred

# Afficher les premières lignes avec les prédictions

# Sauvegarder le fichier avec les prédictions
# Sentiment_test.to_csv("Sentiment_test_with_predictions.csv", index=False)

Extraction des embeddings: 100%|██████████| 300/300 [00:08<00:00, 33.76it/s]


## Créer le DataFrame de soumission

In [63]:
sentiment_submission = pd.DataFrame({
    'ID': Sentiment_test['ID'],
    'Response': Sentiment_test['predictions']
})

In [64]:
print(sentiment_submission.head())
print(sentiment_submission.shape)

                                    ID  Response
0    ID_f3c74c7b_sentiment_test__hausa         2
1    ID_aad19dbf_sentiment_test__hausa         1
2    ID_f6de0381_sentiment_test__hausa         1
3  ID_cbec84fe_sentiment_test__swahili         2
4    ID_885caf5c_sentiment_test__hausa         0
(300, 2)


# AfriXNLI
## Charger les données

In [65]:
XNLI_train = pd.read_parquet("hf://datasets/lelapa/XNLITrain/data/train-00000-of-00001.parquet")
XNLI_test = pd.read_parquet("hf://datasets/lelapa/XNLITest/data/train-00000-of-00001.parquet")

## Supprimer la colonne `targets` dans XNLI_test

In [66]:
if 'targets' in XNLI_test.columns:
    print(f"Suppression de la colonne 'targets' contenant {XNLI_test['targets'].isna().sum()} valeurs manquantes.")
    XNLI_test = XNLI_test.drop(columns=['targets'])

Suppression de la colonne 'targets' contenant 300 valeurs manquantes.


## Fonction pour extraire les embeddings

In [67]:
def get_embeddings(premises, hypotheses, model, tokenizer, device):
    embeddings = []
    for premise, hypothesis in tqdm(zip(premises, hypotheses), desc="Extraction des embeddings", total=len(premises)):
        text = f"{premise} [SEP] {hypothesis}"  # Combiner la prémisse et l'hypothèse
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Utiliser la dernière couche cachée moyenne comme embedding
            hidden_states = outputs.hidden_states[-1]
            sentence_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(sentence_embedding)
    return embeddings

In [68]:
# Extraire les embeddings pour les données d'entraînement
X_train = get_embeddings(XNLI_train['premise'], XNLI_train['inputs'], model, tokenizer, device)
y_train = XNLI_train['targets']

# Extraire les embeddings pour les données de test
X_test = get_embeddings(XNLI_test['premise'], XNLI_test['inputs'], model, tokenizer, device)

# Entraîner un modèle de classification (Logistic Regression)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Évaluer les performances sur les données d'entraînement (ou validation, si applicable)
y_train_pred = clf.predict(X_train)
print("Rapport de classification (Train):")
print(classification_report(y_train, y_train_pred))
print(f"F1-Score (Train): {f1_score(y_train, y_train_pred, average='macro')}")

# Faire des prédictions sur les données de test
y_test_pred = clf.predict(X_test)

Extraction des embeddings: 100%|██████████| 300/300 [00:13<00:00, 22.85it/s]


Rapport de classification (Train):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       134
           1       1.00      1.00      1.00       132
           2       1.00      1.00      1.00       134

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

F1-Score (Train): 1.0


## Créer un DataFrame pour les prédictions

In [69]:
xnli_submission = pd.DataFrame({
    'ID': XNLI_test['ID'],
    'Response': y_test_pred
})

In [70]:
print(xnli_submission.head())
print(xnli_submission.shape)

                               ID  Response
0  ID_f373d29e_test__afrixnli_hau         1
1  ID_31e0a793_test__afrixnli_hau         0
2  ID_4c44b62a_test__afrixnli_hau         0
3  ID_d3568f00_test__afrixnli_swa         1
4  ID_c1c6ab23_test__afrixnli_hau         1
(300, 2)


#Traduction automatique
## Charger les données

In [71]:
MT_train = pd.read_parquet("hf://datasets/lelapa/MTTrain/data/train-00000-of-00001.parquet")
MT_test = pd.read_parquet("hf://datasets/lelapa/MTTest/data/train-00000-of-00001.parquet")

In [72]:
# Supprimer la colonne `targets` dans MT_test
if 'targets' in MT_test.columns:
    print(f"Suppression de la colonne 'targets' contenant {MT_test['targets'].isna().sum()} valeurs manquantes.")
    MT_test = MT_test.drop(columns=['targets'])

Suppression de la colonne 'targets' contenant 300 valeurs manquantes.


## Fonction pour générer les traductions

In [73]:
def generate_translation(texts, model, tokenizer, device, max_length=256):
    translations = []
    for text in tqdm(texts, desc="Génération des traductions"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model.generate(inputs['input_ids'], max_length=max_length, num_beams=5, early_stopping=True)
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translations.append(translation)
    return translations

In [ ]:
# Générer les traductions pour les données d'entraînement (inputs -> prédictions)
train_inputs = MT_train['inputs']
train_targets = MT_train['targets']  # Cibles pour le calcul du F-score

train_predictions = generate_translation(train_inputs, model, tokenizer, device)

# Évaluer avec le CHaRacter-level F-score
char_fscore = evaluate.load("chrf")
score = char_fscore.compute(predictions=train_predictions, references=train_targets)
print(f"CHaRacter-level F-score sur les données d'entraînement : {score['score']}")

# Générer les traductions pour les données de test
test_inputs = MT_test['inputs']
test_predictions = generate_translation(test_inputs, model, tokenizer, device)

Génération des traductions:   0%|          | 0/600 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Génération des traductions:   0%|          | 1/600 [00:04<43:07,  4.32s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Génération des traductions:   0%|          | 2/600 [00:09<45:57,  4.61s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Génération des traductions:   0%|         

## Créer un DataFrame pour les prédictions

In [ ]:
mt_submission = pd.DataFrame({
    'ID': MT_test['ID'],
    'Translation': test_predictions
})

In [ ]:
print(mt_submission.head())
print(mt_submission.shape)

## `Final Submission`

In [ ]:
final_submission = pd.concat([sentiment_submission, xnli_submission, mt_submission], axis=0, ignore_index=True)
final_submission.to_csv("final_submission.csv", index=False)
print("Le fichier final_submission.csv a été créé avec succès.")

---

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.4 MB/s eta 0:00:00
